In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
!pip install faker
from faker import Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.3 MB/s eta 0:00:00


#`df_personal`

## Data Wrangling

### Gathering Data

In [ ]:
# Initialize Faker for Indonesian locale
fake = Faker('id_ID')

# --- Configuration for Personal Finance Dataset ---
personal_categories = {
    "Makanan & Minuman": [
        "Beli makan siang", "Kopi", "Makan malam restoran", "Jajan", "Belanja dapur"
    ],
    "Transportasi": [
        "Bensin", "Tiket KRL", "Ojek online", "Taksi"
    ],
    "Tempat Tinggal": [
        "Bayar kos", "Iuran lingkungan", "Service AC kos"
    ],
    "Belanja Bulanan": [
        "Belanja kebutuhan pokok", "Pakaian baru", "Perlengkapan rumah"
    ],
    "Pulsa & Internet": [
        "Pulsa prabayar", "Paket data", "Tagihan internet"
    ],
    "Hiburan": [
        "Tiket bioskop", "Langganan streaming", "Konser", "Liburan"
    ],
    "Gaji": [
        "Gaji bulanan", "Bonus"
    ],
    "Transfer Teman/Keluarga": [
        "Transfer teman", "Kirim uang keluarga"
    ],
    "Pembayaran Langganan": [
        "Netflix", "Spotify Premium", "Gym membership", "Cloud storage"
    ]
}
personal_transaction_types = {
    "Pemasukan": ["Gaji", "Transfer Teman/Keluarga"],
    "Pengeluaran": ["Makanan & Minuman", "Transportasi", "Tempat Tinggal", "Belanja Bulanan",
                    "Pulsa & Internet", "Hiburan", "Pembayaran Langganan", "Transfer Teman/Keluarga"]
}
personal_payment_methods = {
    "Tunai": ["Makanan & Minuman", "Transportasi", "Tempat Tinggal", "Belanja Bulanan"],
    "Kartu Debit": ["Gaji", "Transfer Teman/Keluarga","Makanan & Minuman", "Transportasi", "Tempat Tinggal", "Belanja Bulanan",
                    "Pulsa & Internet", "Hiburan", "Pembayaran Langganan", "Transfer Teman/Keluarga"],
    "Kartu Kredit": ["Makanan & Minuman", "Belanja Bulanan", "Pulsa & Internet", "Hiburan", "Pembayaran Langganan"],
    "E-Wallet": ["Gaji", "Transfer Teman/Keluarga","Makanan & Minuman", "Transportasi", "Tempat Tinggal", "Belanja Bulanan",
                 "Pulsa & Internet", "Hiburan", "Pembayaran Langganan", "Transfer Teman/Keluarga"],
    "Transfer Bank": ["Gaji", "Transfer Teman/Keluarga","Makanan & Minuman", "Transportasi", "Tempat Tinggal", "Belanja Bulanan",
                      "Pulsa & Internet", "Hiburan", "Pembayaran Langganan", "Transfer Teman/Keluarga"]
}
personal_num_rows = 3500 # Minimal 3000 baris

In [ ]:
# --- Helper function untuk generate amount realistis ---
def generate_personal_amount(category, description):

    if category == "Makanan & Minuman":
        if description == "Beli makan siang":
            return random.randint(10000, 200000)

        elif description in ["Makan malam restoran", "Belanja dapur"]:
            return random.randint(250000, 750000)

        elif description in ["Kopi", "Jajan"]:
            return random.randint(10000, 70000)

    elif category == "Transportasi":
        return random.randint(10000, 100000)

    elif category == "Tempat Tinggal":
        if description == "Bayar kos":
            return random.randint(800000, 1000000)

        elif description in ["Iuran lingkungan", "Service AC kos"]:
            return random.randint(50000, 52000)

    elif category == "Belanja Bulanan":
        return random.randint(200000, 500000)

    elif category == "Pulsa & Internet":
        return random.randint(50000, 150000)

    elif category == "Hiburan":
        if description in ["Tiket bioskop", "Langganan streaming"]:
            return random.randint(30000, 50000)

        elif description in ["Konser", "Liburan"]:
            return random.randint(300000, 500000)

    elif category == "Gaji":
        if description == "Gaji bulanan":
            return random.randint(6000000, 6500000)

        elif description == "Bonus":
            return random.randint(1000000, 3000000)

    elif category == "Transfer Teman/Keluarga":
        return random.randint(50000, 300000)

    elif category == "Pembayaran Langganan":
        if description in ["Netflix", "Spotify Premium"]:
            return random.randint(20000, 30000)

        elif description == "Gym membership":
            return random.randint(200000, 201000)

        elif description == "Cloud storage":
            return random.randint(50000, 55000)


In [ ]:
# --- Generate Personal Finance Dataset Realistis ---
def generate_realistic_personal_dataset(num_rows):

    data = []

    start_date = datetime.now() - timedelta(days=730)
    end_date = datetime.now()

    # generate daftar bulan 2 tahun terakhir
    months = pd.period_range(
        start=start_date.strftime('%Y-%m'),
        end=end_date.strftime('%Y-%m'),
        freq='M'
    )

    transaction_id = 1

    # pastikan tiap bulan ada pemasukan dan pengeluaran serta pengeluaran < pemasukan

    for month in months:

        monthly_income_total = 0
        monthly_expense_total = 0

        # generate pemasukan
        income_count = random.randint(1, 3)

        for _ in range(income_count):

            category = random.choice(
                personal_transaction_types["Pemasukan"]
            )

            description = random.choice(
                personal_categories[category]
            )

            amount = generate_personal_amount(category, description)

            monthly_income_total += amount

            payment_method = random.choice([
                "Kartu Debit",
                "E-Wallet",
                "Transfer Bank"
            ])

            transaction_date = fake.date_between_dates(
                date_start=month.start_time.date(),
                date_end=month.end_time.date()
            )

            data.append({
                "transaction_id": transaction_id,
                "transaction_date": transaction_date.strftime("%Y-%m-%d"),
                "description": description,
                "category": category,
                "transaction_type": "Pemasukan",
                "amount": amount,
                "payment_method": payment_method
            })

            transaction_id += 1

        # generate pengeluaran
        while monthly_expense_total < monthly_income_total * 0.8:

            category = random.choice(
                personal_transaction_types["Pengeluaran"]
            )

            description = random.choice(
                personal_categories[category]
            )

            amount = generate_personal_amount(category, description)

            # supaya pengeluaran tidak melebihi pemasukan
            if monthly_expense_total + amount > monthly_income_total:
                continue

            monthly_expense_total += amount

            available_methods = [
                method for method, cats
                in personal_payment_methods.items()
                if category in cats
            ]

            payment_method = random.choice(available_methods)

            transaction_date = fake.date_between_dates(
                date_start=month.start_time.date(),
                date_end=month.end_time.date()
            )

            data.append({
                "transaction_id": transaction_id,
                "transaction_date": transaction_date.strftime("%Y-%m-%d"),
                "description": description,
                "category": category,
                "transaction_type": "Pengeluaran",
                "amount": amount,
                "payment_method": payment_method
            })

            transaction_id += 1

    # tambahkan data tambahan sampai jumlah rows terpenuhi

    while len(data) < num_rows:

        chosen_type = random.choice(
            list(personal_transaction_types.keys())
        )

        category = random.choice(
            personal_transaction_types[chosen_type]
        )

        description = random.choice(
            personal_categories[category]
        )

        amount = generate_personal_amount(category, description)

        available_methods = [
            method for method, cats
            in personal_payment_methods.items()
            if category in cats
        ]

        payment_method = random.choice(available_methods)

        transaction_date = fake.date_between_dates(
            date_start=start_date,
            date_end=end_date
        )

        data.append({
            "transaction_id": transaction_id,
            "transaction_date": transaction_date.strftime("%Y-%m-%d"),
            "description": description,
            "category": category,
            "transaction_type": chosen_type,
            "amount": amount,
            "payment_method": payment_method
        })

        transaction_id += 1

    df = pd.DataFrame(data[:num_rows])

    return df


# --- Generate dataset ---
df_personal = generate_realistic_personal_dataset(personal_num_rows)

# --- Preview ---
display(df_personal.head())

print(df_personal.info())

,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1,2024-05-17,Bonus,Gaji,Pemasukan,1912165,Kartu Debit
1,2,2024-05-18,Kirim uang keluarga,Transfer Teman/Keluarga,Pemasukan,201838,E-Wallet
2,3,2024-05-01,Ojek online,Transportasi,Pengeluaran,61356,Transfer Bank
3,4,2024-05-12,Perlengkapan rumah,Belanja Bulanan,Pengeluaran,237207,Kartu Kredit
4,5,2024-05-10,Service AC kos,Tempat Tinggal,Pengeluaran,51338,Kartu Debit


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    3500 non-null   int64 
 1   transaction_date  3500 non-null   object
 2   description       3500 non-null   object
 3   category          3500 non-null   object
 4   transaction_type  3500 non-null   object
 5   amount            3500 non-null   int64 
 6   payment_method    3500 non-null   object
dtypes: int64(2), object(5)
memory usage: 191.5+ KB
None


In [ ]:
# --- Save Datasets to CSV ---
df_personal.to_csv("fintrack_personal_dataset.csv", index=False)
print("\nDatasets saved as 'fintrack_personal_dataset.csv'")


Datasets saved as 'fintrack_personal_dataset.csv'


In [ ]:
# Define noise parameters
missing_percentage = 0.01  # ~1% missing values
duplicate_percentage = 0.05 # ~5% duplicate rows
negative_amount_percentage = 0.02 # ~2% negative amounts
inconsistency_percentage = 0.03 # ~3% inconsistent data

# Inconsistency mappings for personal and business datasets
# Key is a description, value is the category it will be inconsistently assigned to.
personal_inconsistency_map = {
    "Beli makan siang": "Transportasi",
    "Kopi": "Pembayaran Langganan",
    "Makan malam restoran": "Pulsa & Internet",
    "Bensin": "Makanan & Minuman",
    "Tiket KRL": "Hiburan",
    "Bayar kos": "Gaji",
    "Netflix": "Makanan & Minuman",
    "Gaji bulanan": "Hiburan",
    "Bonus": "Pembayaran Langganan"
}

def add_noise_to_dataframe(df, inconsistency_map, all_categories_map):
    df_noisy = df.copy()
    num_rows = len(df_noisy)
    num_cols = len(df_noisy.columns)

    print("  Applying noise...")

    # 1. Add Missing Values
    num_missing_values = int(num_rows * num_cols * missing_percentage)
    rows_to_modify = random.choices(range(num_rows), k=num_missing_values)
    cols_to_modify = random.choices(range(num_cols), k=num_missing_values)

    for r, c in zip(rows_to_modify, cols_to_modify):
        df_noisy.iloc[r, c] = np.nan

    # 2. Add Duplicate Data
    num_duplicates_to_add = int(num_rows * duplicate_percentage)
    if num_duplicates_to_add > 0:
        rows_to_duplicate_indices = random.choices(range(num_rows), k=num_duplicates_to_add)
        rows_to_duplicate = df_noisy.iloc[rows_to_duplicate_indices]
        df_noisy = pd.concat([df_noisy, rows_to_duplicate], ignore_index=True)

    # Update num_rows after adding duplicates for subsequent calculations
    num_rows_after_duplicates = len(df_noisy)

    # 3. Add Invalid Data (Negative Amounts)
    # Ensure 'amount' column exists and is numeric before attempting to modify
    if 'amount' in df_noisy.columns and pd.api.types.is_numeric_dtype(df_noisy['amount']):
        num_negative_amounts = int(num_rows_after_duplicates * negative_amount_percentage)
        # Filter for positive amounts to convert to negative
        amount_indices = df_noisy[df_noisy['amount'] > 0].index.tolist()
        if amount_indices:
            indices_to_negate = random.sample(amount_indices, min(num_negative_amounts, len(amount_indices)))
            df_noisy.loc[indices_to_negate, 'amount'] = df_noisy.loc[indices_to_negate, 'amount'] * -1

    # 4. Add Inconsistent Data (Category vs. Description)
    if 'description' in df_noisy.columns and 'category' in df_noisy.columns:
        num_inconsistent_rows = int(num_rows_after_duplicates * inconsistency_percentage)

        # Create a reverse map for efficient lookup of original category by description
        description_to_original_category = {}
        for cat, descs in all_categories_map.items():
            for desc in descs:
                description_to_original_category[desc] = cat

        eligible_indices_for_inconsistency = []
        for idx, row in df_noisy.iterrows():
            description = row['description']
            current_category = str(row['category']) # Convert to string to handle potential NaN from missing values
            # Check if the description is in our inconsistency map and its current category matches its 'original' category
            if description in inconsistency_map and description_to_original_category.get(description) == current_category:
                eligible_indices_for_inconsistency.append(idx)

        if eligible_indices_for_inconsistency:
            indices_to_make_inconsistent = random.sample(
                eligible_indices_for_inconsistency,
                min(num_inconsistent_rows, len(eligible_indices_for_inconsistency))
            )

            for idx in indices_to_make_inconsistent:
                desc = df_noisy.loc[idx, 'description']
                if desc in inconsistency_map:
                    df_noisy.loc[idx, 'category'] = inconsistency_map[desc]

    print("  Noise injection complete.")
    return df_noisy


print("Applying noise to Personal Finance Dataset (df_personal)...")
df_personal_noisy = add_noise_to_dataframe(df_personal, personal_inconsistency_map, personal_categories)
print("\nNoisy Personal Finance Dataset (df_personal_noisy) Head:")
display(df_personal_noisy.head())
print("\nNoisy Personal Finance Dataset (df_personal_noisy) Info:")
display(df_personal_noisy.info())

# Save noisy datasets
df_personal_noisy.to_csv("fintrack_personal_noisy_dataset.csv", index=False)
print("\nNoisy datasets saved as 'fintrack_personal_noisy_dataset.csv'.")

Applying noise to Personal Finance Dataset (df_personal)...
  Applying noise...
  Noise injection complete.

Noisy Personal Finance Dataset (df_personal_noisy) Head:


,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1.0,2024-05-17,Bonus,Gaji,Pemasukan,1912165.0,Kartu Debit
1,2.0,2024-05-18,Kirim uang keluarga,Transfer Teman/Keluarga,Pemasukan,201838.0,E-Wallet
2,3.0,2024-05-01,Ojek online,Transportasi,Pengeluaran,61356.0,Transfer Bank
3,4.0,2024-05-12,Perlengkapan rumah,Belanja Bulanan,Pengeluaran,237207.0,Kartu Kredit
4,5.0,2024-05-10,Service AC kos,Tempat Tinggal,Pengeluaran,51338.0,Kartu Debit



Noisy Personal Finance Dataset (df_personal_noisy) Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3675 entries, 0 to 3674
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    3641 non-null   float64
 1   transaction_date  3637 non-null   object 
 2   description       3638 non-null   object 
 3   category          3643 non-null   object 
 4   transaction_type  3625 non-null   object 
 5   amount            3640 non-null   float64
 6   payment_method    3641 non-null   object 
dtypes: float64(2), object(5)
memory usage: 201.1+ KB


None


Noisy datasets saved as 'fintrack_personal_noisy_dataset.csv'.


# `df_business`

## Data Wrangling

### Gathering Data

In [ ]:
# Initialize Faker for Indonesian locale
fake = Faker('id_ID')

# --- Configuration for Business Finance Dataset ---
business_categories = {
    "Penjualan": [
        "Penjualan produk", "Penjualan jasa", "Pendapatan marketplace",
        "Pendapatan toko offline", "Pendapatan proyek", "Pendapatan grosir"
    ],
    "Modal & Investasi": [
        "Setoran modal pemilik", "Investasi investor", "Pinjaman usaha",
        "Tambahan modal operasional"
    ],
    "Piutang": [
        "Pembayaran piutang pelanggan", "Cicilan pelanggan",
        "Pelunasan invoice", "DP dari pelanggan"
    ],
    "Pembelian Stok": [
        "Pembelian bahan baku", "Pembelian barang dagang",
        "Restock produk", "Pembelian perlengkapan produksi"
    ],
    "Operasional Kantor": [
        "Sewa kantor", "Listrik kantor", "Air kantor",
        "Internet kantor", "ATK", "Kebersihan kantor"
    ],
    "Gaji & Karyawan": [
        "Gaji karyawan", "Bonus karyawan", "THR",
        "Uang lembur", "BPJS Karyawan", "Reimbursement karyawan"
    ],
    "Marketing & Promosi": [
        "Iklan online", "Influencer marketing", "Cetak brosur",
        "Diskon promosi", "Event marketing", "Biaya endorsement"
    ],
    "Transportasi & Logistik": [
        "Ongkos kirim", "Kurir", "Bensin operasional",
        "Sewa kendaraan", "Biaya ekspedisi", "Parkir operasional"
    ],
    "Peralatan & Aset": [
        "Pembelian laptop", "Pembelian printer", "Pembelian mesin produksi",
        "Pembelian furniture kantor", "Perawatan aset"
    ],
    "Software & Langganan": [
        "Langganan software akuntansi", "Cloud storage bisnis",
        "Domain website", "Hosting website", "Tools desain",
        "CRM subscription"
    ],
    "Pajak & Perizinan": [
        "Bayar pajak usaha", "PPh", "PPN", "Biaya izin usaha",
        "Konsultan pajak", "Administrasi legal"
    ],
    "Utang & Cicilan": [
        "Pembayaran utang supplier", "Cicilan pinjaman usaha",
        "Bunga pinjaman", "Pelunasan kredit usaha"
    ],
    "Biaya Bank": [
        "Biaya admin bank", "Biaya transfer", "Biaya payment gateway",
        "Biaya MDR kartu", "Biaya tarik tunai"
    ],
    "Lain-lain": [
        "Biaya tak terduga", "Donasi perusahaan", "Selisih kas",
        "Refund pelanggan", "Koreksi transaksi"
    ]
}

business_transaction_types = {
    "Pemasukan": [
        "Penjualan", "Modal & Investasi", "Piutang"
    ],
    "Pengeluaran": [
        "Pembelian Stok", "Operasional Kantor", "Gaji & Karyawan",
        "Marketing & Promosi", "Transportasi & Logistik", "Peralatan & Aset",
        "Software & Langganan", "Pajak & Perizinan", "Utang & Cicilan",
        "Biaya Bank", "Lain-lain"
    ]
}

business_payment_methods = {
    "Tunai": [
        "Penjualan", "Pembelian Stok", "Operasional Kantor",
        "Transportasi & Logistik", "Lain-lain"
    ],
    "Kartu Debit": [
        "Penjualan", "Pembelian Stok", "Operasional Kantor",
        "Marketing & Promosi", "Transportasi & Logistik",
        "Peralatan & Aset", "Software & Langganan", "Biaya Bank"
    ],
    "Kartu Kredit": [
        "Pembelian Stok", "Marketing & Promosi", "Transportasi & Logistik",
        "Peralatan & Aset", "Software & Langganan", "Operasional Kantor"
    ],
    "E-Wallet": [
        "Penjualan", "Pembelian Stok", "Marketing & Promosi",
        "Transportasi & Logistik", "Software & Langganan",
        "Biaya Bank", "Lain-lain"
    ],
    "Transfer Bank": [
        "Penjualan", "Modal & Investasi", "Piutang",
        "Pembelian Stok", "Operasional Kantor", "Gaji & Karyawan",
        "Marketing & Promosi", "Transportasi & Logistik",
        "Peralatan & Aset", "Software & Langganan", "Pajak & Perizinan",
        "Utang & Cicilan", "Biaya Bank", "Lain-lain"
    ],
    "Payment Gateway": [
        "Penjualan", "Piutang", "Biaya Bank", "Refund pelanggan"
    ],
    "QRIS": [
        "Penjualan", "Piutang", "Biaya Bank"
    ]
}

business_num_rows = 3000  # Minimal 3000 baris

In [ ]:
# --- Helper function untuk generate amount realistis ---
def generate_business_amount(category, description):

    if category == "Penjualan":
        if description in ["Penjualan produk", "Pendapatan toko offline"]:
            return random.randint(50000, 2500000)

        elif description in ["Penjualan jasa", "Pendapatan proyek"]:
            return random.randint(1000000, 25000000)

        elif description == "Pendapatan marketplace":
            return random.randint(100000, 5000000)

        elif description == "Pendapatan grosir":
            return random.randint(3000000, 50000000)

    elif category == "Modal & Investasi":
        if description == "Setoran modal pemilik":
            return random.randint(5000000, 50000000)

        elif description == "Investasi investor":
            return random.randint(25000000, 250000000)

        elif description == "Pinjaman usaha":
            return random.randint(10000000, 100000000)

        elif description == "Tambahan modal operasional":
            return random.randint(2000000, 20000000)

    elif category == "Piutang":
        if description in ["Pembayaran piutang pelanggan", "Pelunasan invoice"]:
            return random.randint(500000, 30000000)

        elif description == "Cicilan pelanggan":
            return random.randint(250000, 5000000)

        elif description == "DP dari pelanggan":
            return random.randint(500000, 15000000)

    elif category == "Pembelian Stok":
        if description in ["Pembelian bahan baku", "Pembelian barang dagang"]:
            return random.randint(500000, 20000000)

        elif description == "Restock produk":
            return random.randint(1000000, 30000000)

        elif description == "Pembelian perlengkapan produksi":
            return random.randint(250000, 10000000)

    elif category == "Operasional Kantor":
        if description == "Sewa kantor":
            return random.randint(2000000, 15000000)

        elif description in ["Listrik kantor", "Air kantor", "Internet kantor"]:
            return random.randint(100000, 1500000)

        elif description in ["ATK", "Kebersihan kantor"]:
            return random.randint(50000, 1000000)

    elif category == "Gaji & Karyawan":
        if description == "Gaji karyawan":
            return random.randint(3000000, 15000000)

        elif description in ["Bonus karyawan", "THR"]:
            return random.randint(1000000, 10000000)

        elif description == "Uang lembur":
            return random.randint(200000, 3000000)

        elif description == "BPJS Karyawan":
            return random.randint(150000, 1500000)

        elif description == "Reimbursement karyawan":
            return random.randint(50000, 2000000)

    elif category == "Marketing & Promosi":
        if description in ["Iklan online", "Influencer marketing", "Event marketing", "Biaya endorsement"]:
            return random.randint(500000, 20000000)

        elif description in ["Cetak brosur", "Diskon promosi"]:
            return random.randint(100000, 5000000)

    elif category == "Transportasi & Logistik":
        if description in ["Ongkos kirim", "Kurir", "Biaya ekspedisi"]:
            return random.randint(10000, 3000000)

        elif description in ["Bensin operasional", "Parkir operasional"]:
            return random.randint(10000, 750000)

        elif description == "Sewa kendaraan":
            return random.randint(300000, 5000000)

    elif category == "Peralatan & Aset":
        if description in ["Pembelian laptop", "Pembelian mesin produksi"]:
            return random.randint(5000000, 50000000)

        elif description in ["Pembelian printer", "Pembelian furniture kantor"]:
            return random.randint(1000000, 15000000)

        elif description == "Perawatan aset":
            return random.randint(250000, 5000000)

    elif category == "Software & Langganan":
        if description in ["Langganan software akuntansi", "Cloud storage bisnis", "Tools desain", "CRM subscription"]:
            return random.randint(50000, 3000000)

        elif description in ["Domain website", "Hosting website"]:
            return random.randint(100000, 2500000)

    elif category == "Pajak & Perizinan":
        if description in ["Bayar pajak usaha", "PPh", "PPN"]:
            return random.randint(500000, 50000000)

        elif description in ["Biaya izin usaha", "Konsultan pajak", "Administrasi legal"]:
            return random.randint(500000, 15000000)

    elif category == "Utang & Cicilan":
        if description in ["Pembayaran utang supplier", "Pelunasan kredit usaha"]:
            return random.randint(1000000, 50000000)

        elif description == "Cicilan pinjaman usaha":
            return random.randint(1000000, 20000000)

        elif description == "Bunga pinjaman":
            return random.randint(100000, 5000000)

    elif category == "Biaya Bank":
        if description in ["Biaya admin bank", "Biaya transfer", "Biaya tarik tunai"]:
            return random.randint(2500, 50000)

        elif description in ["Biaya payment gateway", "Biaya MDR kartu"]:
            return random.randint(10000, 1000000)

    elif category == "Lain-lain":
        if description in ["Biaya tak terduga", "Donasi perusahaan"]:
            return random.randint(100000, 10000000)

        elif description in ["Selisih kas", "Koreksi transaksi"]:
            return random.randint(1000, 500000)

        elif description == "Refund pelanggan":
            return random.randint(50000, 5000000)

In [ ]:
# --- Generate Business Finance Dataset Realistis ---
def generate_realistic_business_dataset(num_rows):

    data = []

    start_date = datetime.now() - timedelta(days=730)
    end_date = datetime.now()

    # generate daftar bulan 2 tahun terakhir
    months = pd.period_range(
        start=start_date.strftime('%Y-%m'),
        end=end_date.strftime('%Y-%m'),
        freq='M'
    )

    transaction_id = 1

    # pastikan tiap bulan ada pemasukan dan pengeluaran
    # serta pengeluaran tidak melebihi pemasukan

    for month in months:

        monthly_income_total = 0
        monthly_expense_total = 0

        # generate pemasukan bisnis
        income_count = random.randint(3, 10)

        for _ in range(income_count):

            category = random.choice(
                business_transaction_types["Pemasukan"]
            )

            description = random.choice(
                business_categories[category]
            )

            amount = generate_business_amount(category, description)

            monthly_income_total += amount

            available_methods = [
                method for method, cats
                in business_payment_methods.items()
                if category in cats
            ]

            payment_method = random.choice(available_methods)

            transaction_date = fake.date_between_dates(
                date_start=month.start_time.date(),
                date_end=month.end_time.date()
            )

            data.append({
                "transaction_id": transaction_id,
                "transaction_date": transaction_date.strftime("%Y-%m-%d"),
                "description": description,
                "category": category,
                "transaction_type": "Pemasukan",
                "amount": amount,
                "payment_method": payment_method
            })

            transaction_id += 1

        # generate pengeluaran bisnis
        while monthly_expense_total < monthly_income_total * 0.8:

            category = random.choice(
                business_transaction_types["Pengeluaran"]
            )

            description = random.choice(
                business_categories[category]
            )

            amount = generate_business_amount(category, description)

            # supaya pengeluaran tidak melebihi pemasukan
            if monthly_expense_total + amount > monthly_income_total:
                continue

            monthly_expense_total += amount

            available_methods = [
                method for method, cats
                in business_payment_methods.items()
                if category in cats
            ]

            payment_method = random.choice(available_methods)

            transaction_date = fake.date_between_dates(
                date_start=month.start_time.date(),
                date_end=month.end_time.date()
            )

            data.append({
                "transaction_id": transaction_id,
                "transaction_date": transaction_date.strftime("%Y-%m-%d"),
                "description": description,
                "category": category,
                "transaction_type": "Pengeluaran",
                "amount": amount,
                "payment_method": payment_method
            })

            transaction_id += 1

    # tambahkan data tambahan sampai jumlah rows terpenuhi

    while len(data) < num_rows:

        chosen_type = random.choice(
            list(business_transaction_types.keys())
        )

        category = random.choice(
            business_transaction_types[chosen_type]
        )

        description = random.choice(
            business_categories[category]
        )

        amount = generate_business_amount(category, description)

        available_methods = [
            method for method, cats
            in business_payment_methods.items()
            if category in cats
        ]

        payment_method = random.choice(available_methods)

        transaction_date = fake.date_between_dates(
            date_start=start_date,
            date_end=end_date
        )

        data.append({
            "transaction_id": transaction_id,
            "transaction_date": transaction_date.strftime("%Y-%m-%d"),
            "description": description,
            "category": category,
            "transaction_type": chosen_type,
            "amount": amount,
            "payment_method": payment_method
        })

        transaction_id += 1

    df = pd.DataFrame(data[:num_rows])

    return df


# --- Generate dataset ---
df_business = generate_realistic_business_dataset(business_num_rows)

# --- Preview ---
display(df_business.head())

print(df_business.info())

,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1,2024-05-27,Pendapatan marketplace,Penjualan,Pemasukan,676749,QRIS
1,2,2024-05-15,Pendapatan proyek,Penjualan,Pemasukan,22241448,Tunai
2,3,2024-05-01,Investasi investor,Modal & Investasi,Pemasukan,85947193,Transfer Bank
3,4,2024-05-05,Penjualan produk,Penjualan,Pemasukan,1278397,Tunai
4,5,2024-05-06,Pendapatan proyek,Penjualan,Pemasukan,10420797,Tunai


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    3000 non-null   int64 
 1   transaction_date  3000 non-null   object
 2   description       3000 non-null   object
 3   category          3000 non-null   object
 4   transaction_type  3000 non-null   object
 5   amount            3000 non-null   int64 
 6   payment_method    3000 non-null   object
dtypes: int64(2), object(5)
memory usage: 164.2+ KB
None


In [ ]:
# --- Save Business Dataset to CSV ---
df_business.to_csv("fintrack_business_dataset.csv", index=False)
print("\nDataset saved as 'fintrack_business_dataset.csv'")


Dataset saved as 'fintrack_business_dataset.csv'


In [ ]:
# Define noise parameters
missing_percentage = 0.01          # ~1% missing values
duplicate_percentage = 0.05        # ~5% duplicate rows
negative_amount_percentage = 0.02  # ~2% negative amounts
inconsistency_percentage = 0.03     # ~3% inconsistent data

# Inconsistency mappings for business dataset
# Key adalah description, value adalah category yang salah / tidak konsisten.
business_inconsistency_map = {
    "Penjualan produk": "Gaji & Karyawan",
    "Penjualan jasa": "Pembelian Stok",
    "Pendapatan marketplace": "Operasional Kantor",
    "Pendapatan toko offline": "Software & Langganan",
    "Pendapatan proyek": "Transportasi & Logistik",
    "Pendapatan grosir": "Biaya Bank",

    "Setoran modal pemilik": "Marketing & Promosi",
    "Investasi investor": "Pembelian Stok",
    "Pinjaman usaha": "Pajak & Perizinan",
    "Tambahan modal operasional": "Utang & Cicilan",

    "Pembayaran piutang pelanggan": "Operasional Kantor",
    "Cicilan pelanggan": "Gaji & Karyawan",
    "Pelunasan invoice": "Biaya Bank",
    "DP dari pelanggan": "Peralatan & Aset",

    "Pembelian bahan baku": "Penjualan",
    "Pembelian barang dagang": "Piutang",
    "Restock produk": "Modal & Investasi",
    "Pembelian perlengkapan produksi": "Penjualan",

    "Sewa kantor": "Penjualan",
    "Listrik kantor": "Piutang",
    "Air kantor": "Modal & Investasi",
    "Internet kantor": "Penjualan",
    "ATK": "Piutang",
    "Kebersihan kantor": "Modal & Investasi",

    "Gaji karyawan": "Penjualan",
    "Bonus karyawan": "Piutang",
    "THR": "Modal & Investasi",
    "Uang lembur": "Penjualan",
    "BPJS Karyawan": "Piutang",
    "Reimbursement karyawan": "Modal & Investasi",

    "Iklan online": "Penjualan",
    "Influencer marketing": "Piutang",
    "Cetak brosur": "Modal & Investasi",
    "Diskon promosi": "Penjualan",
    "Event marketing": "Piutang",
    "Biaya endorsement": "Modal & Investasi",

    "Ongkos kirim": "Penjualan",
    "Kurir": "Piutang",
    "Bensin operasional": "Modal & Investasi",
    "Sewa kendaraan": "Penjualan",
    "Biaya ekspedisi": "Piutang",
    "Parkir operasional": "Modal & Investasi",

    "Pembelian laptop": "Penjualan",
    "Pembelian printer": "Piutang",
    "Pembelian mesin produksi": "Modal & Investasi",
    "Pembelian furniture kantor": "Penjualan",
    "Perawatan aset": "Piutang",

    "Langganan software akuntansi": "Penjualan",
    "Cloud storage bisnis": "Piutang",
    "Domain website": "Modal & Investasi",
    "Hosting website": "Penjualan",
    "Tools desain": "Piutang",
    "CRM subscription": "Modal & Investasi",

    "Bayar pajak usaha": "Penjualan",
    "PPh": "Piutang",
    "PPN": "Modal & Investasi",
    "Biaya izin usaha": "Penjualan",
    "Konsultan pajak": "Piutang",
    "Administrasi legal": "Modal & Investasi",

    "Pembayaran utang supplier": "Penjualan",
    "Cicilan pinjaman usaha": "Piutang",
    "Bunga pinjaman": "Modal & Investasi",
    "Pelunasan kredit usaha": "Penjualan",

    "Biaya admin bank": "Penjualan",
    "Biaya transfer": "Piutang",
    "Biaya payment gateway": "Modal & Investasi",
    "Biaya MDR kartu": "Penjualan",
    "Biaya tarik tunai": "Piutang",

    "Biaya tak terduga": "Penjualan",
    "Donasi perusahaan": "Piutang",
    "Selisih kas": "Modal & Investasi",
    "Refund pelanggan": "Pembelian Stok",
    "Koreksi transaksi": "Gaji & Karyawan"
}


def add_noise_to_dataframe(df, inconsistency_map, all_categories_map):
    df_noisy = df.copy()
    num_rows = len(df_noisy)
    num_cols = len(df_noisy.columns)

    print("  Applying noise...")

    # 1. Add Missing Values
    num_missing_values = int(num_rows * num_cols * missing_percentage)
    rows_to_modify = random.choices(range(num_rows), k=num_missing_values)
    cols_to_modify = random.choices(range(num_cols), k=num_missing_values)

    for r, c in zip(rows_to_modify, cols_to_modify):
        df_noisy.iloc[r, c] = np.nan

    # 2. Add Duplicate Data
    num_duplicates_to_add = int(num_rows * duplicate_percentage)

    if num_duplicates_to_add > 0:
        rows_to_duplicate_indices = random.choices(
            range(num_rows),
            k=num_duplicates_to_add
        )

        rows_to_duplicate = df_noisy.iloc[rows_to_duplicate_indices]
        df_noisy = pd.concat(
            [df_noisy, rows_to_duplicate],
            ignore_index=True
        )

    # Update num_rows after adding duplicates for subsequent calculations
    num_rows_after_duplicates = len(df_noisy)

    # 3. Add Invalid Data, yaitu negative amounts
    # Ensure 'amount' column exists and is numeric before attempting to modify
    if 'amount' in df_noisy.columns and pd.api.types.is_numeric_dtype(df_noisy['amount']):

        num_negative_amounts = int(
            num_rows_after_duplicates * negative_amount_percentage
        )

        # Filter for positive amounts to convert to negative
        amount_indices = df_noisy[df_noisy['amount'] > 0].index.tolist()

        if amount_indices:
            indices_to_negate = random.sample(
                amount_indices,
                min(num_negative_amounts, len(amount_indices))
            )

            df_noisy.loc[indices_to_negate, 'amount'] = (
                df_noisy.loc[indices_to_negate, 'amount'] * -1
            )

    # 4. Add Inconsistent Data, yaitu category vs description
    if 'description' in df_noisy.columns and 'category' in df_noisy.columns:

        num_inconsistent_rows = int(
            num_rows_after_duplicates * inconsistency_percentage
        )

        # Create a reverse map for efficient lookup of original category by description
        description_to_original_category = {}

        for cat, descs in all_categories_map.items():
            for desc in descs:
                description_to_original_category[desc] = cat

        eligible_indices_for_inconsistency = []

        for idx, row in df_noisy.iterrows():
            description = row['description']
            current_category = str(row['category'])

            # Check if description ada di inconsistency_map
            # dan category saat ini masih sesuai category original
            if (
                description in inconsistency_map
                and description_to_original_category.get(description) == current_category
            ):
                eligible_indices_for_inconsistency.append(idx)

        if eligible_indices_for_inconsistency:
            indices_to_make_inconsistent = random.sample(
                eligible_indices_for_inconsistency,
                min(num_inconsistent_rows, len(eligible_indices_for_inconsistency))
            )

            for idx in indices_to_make_inconsistent:
                desc = df_noisy.loc[idx, 'description']

                if desc in inconsistency_map:
                    df_noisy.loc[idx, 'category'] = inconsistency_map[desc]

    print("  Noise injection complete.")

    return df_noisy


print("Applying noise to Business Finance Dataset (df_business)...")

df_business_noisy = add_noise_to_dataframe(
    df_business,
    business_inconsistency_map,
    business_categories
)

print("\nNoisy Business Finance Dataset (df_business_noisy) Head:")
display(df_business_noisy.head())

print("\nNoisy Business Finance Dataset (df_business_noisy) Info:")
display(df_business_noisy.info())

# Save noisy dataset
df_business_noisy.to_csv("fintrack_business_noisy_dataset.csv", index=False)

print("\nNoisy dataset saved as 'fintrack_business_noisy_dataset.csv'.")

Applying noise to Business Finance Dataset (df_business)...
  Applying noise...
  Noise injection complete.

Noisy Business Finance Dataset (df_business_noisy) Head:


,transaction_id,transaction_date,description,category,transaction_type,amount,payment_method
0,1.0,2024-05-27,Pendapatan marketplace,Penjualan,Pemasukan,676749.0,QRIS
1,2.0,2024-05-15,Pendapatan proyek,Penjualan,Pemasukan,22241448.0,Tunai
2,3.0,2024-05-01,Investasi investor,Modal & Investasi,Pemasukan,85947193.0,Transfer Bank
3,NaN,2024-05-05,Penjualan produk,Penjualan,Pemasukan,1278397.0,Tunai
4,5.0,2024-05-06,Pendapatan proyek,Penjualan,Pemasukan,10420797.0,NaN



Noisy Business Finance Dataset (df_business_noisy) Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    3122 non-null   float64
 1   transaction_date  3112 non-null   object 
 2   description       3115 non-null   object 
 3   category          3121 non-null   object 
 4   transaction_type  3119 non-null   object 
 5   amount            3120 non-null   float64
 6   payment_method    3115 non-null   object 
dtypes: float64(2), object(5)
memory usage: 172.4+ KB


None


Noisy dataset saved as 'fintrack_business_noisy_dataset.csv'.
